# Libaries

In [30]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [31]:
import pandas as pd
import numpy as np

from src.benchmark import (
    ElasticityConfig,
    ModelingDatasetBuilder,
    BenchmarkFairFormulaBuilder,
    ElasticityBenchmarkFairPipeline,
)

from src.dominick import DominickDataLoader
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [32]:
TRAIN_FRAC = 0.8
N_FOLDS = 5
N_BOOTSTRAP = 20

# Loader

In [33]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()

print(f"Dataset shape: {df.shape}")
print(f"N semanas: {df['week_id'].nunique()}")

Dataset shape: (463722, 30)
N semanas: 302


# Config + Pipeline

In [34]:
config = ElasticityConfig(csv_path="elasticity_dataset.csv")
splitter = TemporalSplitter(week_col="week_id")

pipeline = ElasticityBenchmarkFairPipeline(
    config=config,
    dataset_builder=ModelingDatasetBuilder(config),
    formula_builder=BenchmarkFairFormulaBuilder(config),
)

# Evaluation K-fold

In [35]:
fold_results = []
for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(df, N_FOLDS)):
    results = pipeline.run(train_df=train_fold, val_df=val_fold)
    results["fold"] = fold_idx
    fold_results.append(results)

all_folds = pd.concat(fold_results, ignore_index=True)

generalization_cols = ["store_code", "upc_code", "fold", "elasticity", "mae_val", "rmse_val", "r2_val"]
ok_folds = all_folds[all_folds["status"] == "ok"].copy()
benchmark_generalization_folds = ok_folds[generalization_cols].copy()

print(f"Generalización ({len(fold_results)} folds): {len(benchmark_generalization_folds)} filas (store, upc, fold)")

Generalización (5 folds): 7090 filas (store, upc, fold)


# Evaluation Bootstrap

In [36]:
train_df, val_df = splitter.single_split(df, train_frac=TRAIN_FRAC)
train_weeks = sorted(train_df["week_id"].unique())
sampler = BlockBootstrapSampler(week_col="week_id", block_size=4, rng=np.random.default_rng(42))

bootstrap_results = []
for b in range(N_BOOTSTRAP):
    train_bs = sampler.sample(train_df, train_weeks)
    res = pipeline.run(train_df=train_bs, val_df=val_df)
    res["bootstrap_run"] = b
    bootstrap_results.append(res)

all_bootstrap = pd.concat(bootstrap_results, ignore_index=True)

bootstrap_raw_cols = ["store_code", "upc_code", "bootstrap_run", "elasticity", "mae_val", "rmse_val", "r2_val"]
benchmark_elasticity_bootstrap_raw = all_bootstrap[all_bootstrap["status"] == "ok"][bootstrap_raw_cols].copy()

ok_bs = all_bootstrap[all_bootstrap["status"] == "ok"].copy()
def q025(x): return np.percentile(x, 2.5)
def q975(x): return np.percentile(x, 97.5)

benchmark_elasticities_bootstrap_summary = (
    ok_bs
    .groupby(["store_code", "upc_code"])
    .agg(
        elasticity_mean=("elasticity", "mean"),
        elasticity_std=("elasticity", "std"),
        elasticity_ci_low=("elasticity", q025),
        elasticity_ci_high=("elasticity", q975),
        mae_val_mean=("mae_val", "mean"),
        mae_val_std=("mae_val", "std"),
        rmse_val_mean=("rmse_val", "mean"),
        rmse_val_std=("rmse_val", "std"),
        r2_val_mean=("r2_val", "mean"),
        r2_val_std=("r2_val", "std"),
    )
    .reset_index()
)

print(f"Bootstrap raw: {len(benchmark_elasticity_bootstrap_raw)} filas")
print(f"Bootstrap summary: {len(benchmark_elasticities_bootstrap_summary)} series ok")

Bootstrap raw: 8743 filas
Bootstrap summary: 448 series ok


# Results

In [37]:
display(benchmark_generalization_folds.head(10))
display(benchmark_generalization_folds[["elasticity", "mae_val", "rmse_val", "r2_val"]].describe())

,store_code,upc_code,fold,elasticity,mae_val,rmse_val,r2_val
0,5,3410015306,0,-6.364375,0.457436,0.666143,0.101720
1,5,3410017306,0,-2.419663,0.912833,1.107612,-2.398609
2,5,3410057306,0,-3.864755,0.438223,0.554538,-0.685190
3,5,5230000035,0,-16.851507,0.993298,1.371022,-3.284946
4,5,5230000240,0,-6.082512,1.373955,1.608924,-5.016006
5,5,7289000011,0,7.588301,1.038279,1.266727,-3.670802
6,8,1820000784,0,-4.707357,0.575803,0.732174,0.183999
7,8,1820000987,0,-3.949503,0.596609,0.786385,-0.080267
8,8,1820011047,0,-5.701979,0.576921,0.734758,0.100541
9,8,1820011168,0,-6.639618,0.974525,1.169730,-0.578519


,elasticity,mae_val,rmse_val,r2_val
count,7090.000000,7090.000000,7090.000000,7.010000e+03
mean,-3.629566,0.777116,0.924836,-1.043455e+27
std,7.340476,1.807656,2.024726,7.823811e+28
min,-134.011920,0.008004,0.008004,-6.499783e+30
25%,-5.271154,0.433483,0.543055,-9.071850e-01
50%,-3.860343,0.544176,0.671634,-2.223873e-01
75%,-2.358891,0.685885,0.833443,1.667209e-01
max,404.855722,49.953192,55.524918,9.166543e-01


In [38]:
display(benchmark_elasticity_bootstrap_raw.head(10))
display(benchmark_elasticities_bootstrap_summary.head(10))
display(benchmark_elasticities_bootstrap_summary[["elasticity_mean", "elasticity_std", "elasticity_ci_low", "elasticity_ci_high"]].describe())

,store_code,upc_code,bootstrap_run,elasticity,mae_val,rmse_val,r2_val
0,5,5230000035,0,3.702780,0.581912,0.698496,-0.284105
1,5,7289000011,0,2.826643,0.351033,0.457049,0.556837
2,8,1820000784,0,-4.563238,0.515019,0.703928,-0.025925
3,8,3410010505,0,-2.814721,0.517562,0.649902,-0.343690
4,8,5230000035,0,0.136377,0.573995,0.786756,-0.099401
5,8,7199077005,0,-2.490401,0.435489,0.557617,0.267154
6,8,7289000011,0,1.219401,0.707813,0.916784,-0.783549
7,8,8066095605,0,-7.186834,1.121888,1.121888,NaN
9,9,1820000784,0,-4.618333,1.526223,1.759754,-5.908170
10,9,3410010505,0,-1.993350,0.398593,0.481868,0.003362


,store_code,upc_code,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,5,5230000035,-1.809606,4.032201,-11.143212,3.049801,0.710845,0.300590,0.826524,0.303862,-1.028836,1.956233
1,5,7289000011,0.465335,2.854957,-3.681604,5.909174,0.472783,0.076279,0.567166,0.089738,0.301339,0.232127
2,8,1820000784,-4.136866,0.674723,-5.004429,-2.810630,0.525800,0.032478,0.704684,0.035545,-0.030615,0.104789
3,8,3410010505,-3.208125,0.518591,-4.048412,-2.312397,0.611561,0.061823,0.743415,0.066964,-0.771743,0.326934
4,8,5230000035,-1.966848,1.633472,-5.056889,0.876095,0.570293,0.043468,0.736381,0.041872,0.033920,0.111380
5,8,7199077005,-2.663010,0.382353,-3.334465,-2.058325,0.559619,0.086628,0.708016,0.088339,-0.198955,0.296907
6,8,7289000011,-0.352688,2.028672,-4.046766,2.796173,0.642453,0.085843,0.810102,0.092578,-0.409893,0.334138
7,8,8066095605,-10.914787,4.178355,-18.981404,-5.644652,0.477763,0.258606,0.477763,0.258606,NaN,NaN
8,9,1820000784,-4.779381,0.763686,-6.252482,-3.563160,1.366934,0.152470,1.568917,0.165573,-4.549192,1.161805
9,9,3410010505,-2.868036,0.441211,-3.448344,-2.089252,0.399908,0.042517,0.493739,0.043326,-0.054004,0.189862


,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high
count,448.000000,448.000000,448.000000,448.000000
mean,-3.076262,2.851329,-7.831730,1.156324
std,4.373996,10.413552,24.002990,10.944642
min,-59.675109,0.218163,-485.996036,-7.118403
25%,-4.684398,0.672843,-7.198199,-2.751750
50%,-3.154876,1.056229,-5.556103,-0.957903
75%,-1.705608,2.174576,-3.782886,1.433278
max,29.978882,194.108645,20.124252,116.176923


In [39]:
display(all_folds["status"].value_counts(dropna=False))
display(all_bootstrap["status"].value_counts(dropna=False))

status
ok                          7090
insufficient_train_obs       315
no_price_variation_train     291
Name: count, dtype: int64

status
ok                          8743
no_price_variation_train    1157
insufficient_train_obs       454
Name: count, dtype: int64

# Save

In [41]:
benchmark_generalization_folds.to_csv("../data/benchmark_generalization_folds.csv", index=False)
benchmark_elasticity_bootstrap_raw.to_csv("../data/benchmark_elasticity_bootstrap_raw.csv", index=False)
benchmark_elasticities_bootstrap_summary.to_csv("../data/benchmark_elasticities_bootstrap_summary.csv", index=False)
ok_folds.to_csv("../data/benchmark_kfold_raw.csv", index=False)
ok_bs.to_csv("../data/benchmark_bootstrap_raw.csv", index=False)

print("Guardado:")
print("  - benchmark_generalization_folds.csv (generalización temporal raw)")
print("  - benchmark_elasticity_bootstrap_raw.csv (bootstrap raw)")
print("  - benchmark_elasticities_bootstrap_summary.csv (bootstrap summary)")
print("  - benchmark_kfold_raw.csv (kfold raw)")
print("  - benchmark_bootstrap_raw.csv (bootstrap raw)")

Guardado:
  - benchmark_generalization_folds.csv (generalización temporal raw)
  - benchmark_elasticity_bootstrap_raw.csv (bootstrap raw)
  - benchmark_elasticities_bootstrap_summary.csv (bootstrap summary)
  - benchmark_kfold_raw.csv (kfold raw)
  - benchmark_bootstrap_raw.csv (bootstrap raw)
